# Data Exploration
Python notebook to get familiar with the data
</br>
</br>
Kerim Atak (kerim.atak@univie.ac.at)

## Raw Data

In [ ]:
import os
import pandas as pd
import numpy as np
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import json
import os
from session_visualization import generate_interactive_session_plot

%load_ext autoreload
%autoreload 2

print("Current OS Path:", os.getcwd())

In [ ]:
data_session = "JPAS_0023_20230922"

In [ ]:
# import cluster_info.tsv
## contains information about each cluster (neuron), including quality metrics
## NOTE: This is what will be used to filter good units for NC-MCM/BunDLe-Net analysis
df = pd.read_csv(os.path.join(data_session, "cluster_info.tsv"), sep="\t")

# filter column "group" for value "good" to keep only good units
df = df[df["group"] == "good"]

print(df.head())
print("DataFrame Shape:", df.shape)

In [ ]:
# import channel_map.npy
## contains the mapping of channels to their physical locations on the probe ... probably
channel_map = np.load(os.path.join(data_session, "channel_map.npy"), allow_pickle=True)
print("Channel Map Shape:", channel_map.shape)
print("Channel Map Data:", channel_map)

In [ ]:
# import channel_positions.npy
## contains the physical positions of each channel on the probe ... probably
channel_positions = np.load(os.path.join(data_session, "channel_positions.npy"), allow_pickle=True)
print("Channel Positions Shape:", channel_positions.shape)
print("Channel Positions Data:", channel_positions)

In [ ]:
# import spike_times.npy
## contains the timestamps of each spike event of all neurons
## NOTE: This is what will be used for NC-MCM/BunDLe-Net analysis
## NOTE: spike_times, spike_clusters, and spike_templates are aligned by index
spike_times = np.load(os.path.join(data_session, "spike_times.npy"))
print("Spike times shape:", spike_times.shape)
print("First 10 spike times:", spike_times[:10])
print("First spike time:", np.min(spike_times))
print("Last spike time:", np.max(spike_times))

# plot histogramm of spike times
import plotly.express as px
fig = px.histogram(spike_times, nbins=100, title="Spike Times Histogram", labels={"value": "Time (samples)", "count": "Number of Spikes"})
fig.show()



In [ ]:
# import spike_clusters.npy
## assigns each spike to a cluster (neuron)
## NOTE: This is what will be used for NC-MCM/BunDLe-Net analysis
spike_clusters = np.load(os.path.join(data_session, "spike_clusters.npy"))
print("Spike clusters shape:", spike_clusters.shape)
print("First 10 spike clusters:", spike_clusters[:10])

In [ ]:
# import spike_templates.npy
## output file from Kilosort, assigns each spike to a template rather than a cluster (neuron), needs further processing/merging to get neuron assignments
spike_templates = np.load(os.path.join(data_session, "spike_templates.npy"))
print("Spike templates shape:", spike_templates.shape)
print("First 10 spike templates:", spike_templates[:10])

In [ ]:
# import spike_times_milliseconds_sync_to_behav_JPAS_0023_20230922.npy
## contains spike times in milliseconds, synchronized to behavioral data (the time values are the same as behavioral time values in metrics.json)
spike_times_ms_sync_to_behav = np.load(os.path.join(data_session, "spike_times_milliseconds_sync_to_behav.npy"))
print("Spike times ms shape:", spike_times_ms_sync_to_behav.shape)
print("First 10 spike times ms:", spike_times_ms_sync_to_behav[:10])

In [ ]:
# import params.py, read as txt and print content
## contains metadata about the recording session
## parse n_channels_dat, offset, sample_rate, dtype, hp_filtered from params.py
## NOTE: Only sample_rate is interessting for now, which is 32050.755862876373 Hz
with open(os.path.join(data_session, "params.py"), "r") as f:
    params_content = f.read()
    print("Params.py content:\n", params_content)
    n_channels_dat = params_content.split("n_channels_dat = ")[1].split("\n")[0]
    offset = params_content.split("offset = ")[1].split("\n")[0]
    sample_rate = params_content.split("sample_rate = ")[1].split("\n")[0]
    dtype = params_content.split("dtype = '")[1].split("'")[0]
    hp_filtered = params_content.split("hp_filtered = ")[1].split("\n")[0]

print(f"n_channels_dat: {n_channels_dat}, offset: {offset}, sample_rate: {sample_rate}, dtype: {dtype}, hp_filtered: {hp_filtered}")

In [ ]:
# import metrics.json as dict and print keys
## contains various metrics about the recording session like 
with open(os.path.join(data_session, "metrics.json"), "r") as f:
    metrics = json.load(f)
    print("Metrics keys:", metrics.keys())

In [ ]:
def _metrics_json_overview(metrics_dict, showcase_num=5):
    # Print an overview of the metrics.json content
    for key, value in metrics_dict.items():
        print(f"{key}: {type(value)} - {value if isinstance(value, (int, float, str)) else '...'}")
        # if key is "metrics", print the keys inside metrics
        if key == "metrics":
            for metric_key in value.keys():
                print(f"  - {metric_key}")
    
    print()
    
    # experiment data
    print("="*15 + " Experiment Data " + "="*15)
    print(json.dumps(metrics_dict.get("experiment data", {}), indent=4))
    print() 
    
    # performance
    print("="*15 + " Performance " + "="*15)
    print(json.dumps(metrics_dict.get("performance", {}), indent=4))
    print()
    
    # metrics - blocks
    print("="*15 + " Metrics - Blocks " + "="*15)
    # num of blocks
    print("Number of blocks:", len(metrics_dict.get("metrics", {}).get("blocks", {})))
    print(json.dumps(metrics_dict.get("metrics", {}).get("blocks", {}), indent=4))
    print() 
    
    # metrics - trials
    print("="*15 + " Metrics - Trials " + "="*15)
    # num of trials
    trials = metrics_dict.get("metrics", {}).get("trials", [])
    print("Number of trials:", len(trials))
    print(f"First {showcase_num} trials:")
    print(json.dumps(trials[:showcase_num], indent=4))  # Print only the first 5 trials
    print("...")
    
    # metrics - states
    print("="*15 + " Metrics - States " + "="*15)
    states = metrics_dict.get("metrics", {}).get("states", [])
    print("Number of states:", len(states))
    print(json.dumps(states[:showcase_num], indent=4))  # Print only the
    print("...")
    
    # kayeton cam (t ms/#frame/vid time)
    print("="*15 + " Kayeton Cam Data " + "="*15)
    kayeton_cam = metrics_dict.get("kayeton cam (t ms/#frame/vid time)", {})
    print("Number of kayeton cam entries:", len(kayeton_cam))
    # compute average t ms using the first element of each kayeton_cam entry
    t_ms_vals = []
    for entry in kayeton_cam:
        t_ms_vals.append(float(entry[0]))
    intervals = np.diff(t_ms_vals)
    mean_interval = float(np.mean(intervals))
    std_interval = float(np.std(intervals))
    print(f"Average interval between consecutive 't ms': {mean_interval:.3f} ms (±{std_interval:.3f} ms std)")
    print(json.dumps(kayeton_cam[:showcase_num], indent=4))
    print("...")
    
    # wheel
    print("="*15 + " Wheel Data " + "="*15)
    wheel = metrics_dict.get("wheel", {})
    print("Number of wheel entries:", len(wheel))
    print(json.dumps(wheel[:showcase_num], indent=4))
    print("...")

    # clock 100ms
    print("="*15 + " Clock 100ms Data " + "="*15)
    clock_100ms = metrics_dict.get("clock 100ms", {})
    print("Number of clock 100ms entries:", len(clock_100ms))
    print(json.dumps(clock_100ms[:showcase_num], indent=4))
    print("...")
    
    # clock 5min
    print("="*15 + " Clock 5min Data " + "="*15)
    clock_5min = metrics_dict.get("clock 5min", {})
    print("Number of clock 5min entries:", len(clock_5min))
    print(json.dumps(clock_5min[:showcase_num], indent=4))
    print("...")
    
    
_metrics_json_overview(metrics)

In [ ]:
# Generate interactive plot
summary = generate_interactive_session_plot(data_session, output_filename='interactive_trial_plot.html', window=10)

## Processed Data

In [ ]:
from dataset import BanditTaskNeuroPixelsDataset
import plotly.graph_objects as go
import os
import numpy as np
import plotly.express as px
import pandas as pd
from ncmcm.visualisers.neuronal_behavioural import plotting_neuronal_behavioural_plotly
from ncmcm.visualisers.behavioural_discrete import plot_behavior_state_lengths_boxplot, plot_behavior_state_sample_frequencies_barchart, plot_behavior_state_timeline


%load_ext autoreload
%autoreload 2

# go to folder <
old_path = os.getcwd()
os.chdir('/home/kerim/Projects/Neural Algorithms/NC-MCM/')
os.chdir(old_path)

print("Current OS Path:", os.getcwd())

In [ ]:
ds = BanditTaskNeuroPixelsDataset(data_path="JPAS_0023_20230922", 
                                  downsample_fs=25, 
                                  downsample_method='gaussian',
                                  state_transitions=BanditTaskNeuroPixelsDataset.HOLD_TO_CHOOSING_TRANSITIONS,
                                  gaussian_sigma_ms=25.0,
                                  normalize_method='minmax_global'
                                )

In [ ]:
ds.check_state_transitions(ds.DEFAULT_TRANSITION_MAP)

In [ ]:
print("Frequency: ", ds.fs)
print("Recording in mins: ", ds.get_recording_length_mins())

In [ ]:
ds.x

In [ ]:
ds.b

In [ ]:
ds.b_continuous

In [ ]:
np.unique(ds.b)

In [ ]:
ds.b_labels_dict

In [ ]:
_ = plot_behavior_state_sample_frequencies_barchart(ds.b, ds.b_labels_dict, color_map=ds.get_color_map_for_plotting(), title="Behavioral State Frequencies")

In [ ]:
# Full timeline
fig = plot_behavior_state_timeline(
    ds.b, 
    ds.b_labels_dict, 
    sampling_frequency=ds.fs,
    # convert_to_seconds=True,
    color_map=ds.get_color_map_for_plotting(),
    title="Behavioral State Timeline (Full Recording)"
)

In [ ]:
ds.b.toarray().flatten()

In [ ]:
# fig = plotting_neuronal_behavioural_plotly(ds.x.T.toarray(), ds.b.toarray().flatten(), b_names=list(ds.b_labels_dict.values()))
## export as html
# fig.write_html("neuronal_behavioural_plot.html")

In [ ]:
_ = plot_behavior_state_lengths_boxplot(
    behavior_data=ds.b,
    state_labels_dict=ds.b_labels_dict,
    sampling_frequency=ds.fs,
    show_fig=True,
    convert_to_seconds=True,
    color_map=ds.get_color_map_for_plotting(),
    title="Behavioral State Segment Length Distributions"
)

In [ ]:
# plot b_continuous over time
fig = px.line(x=np.arange(ds.b_continuous.shape[0]), y=ds.b_continuous, labels={'x': 'Time', 'y': 'b_continuous'}, title='Continuous Behavioral Variable Over Time')
fig.show()

In [ ]:
# plot time series ds.trial_indices
fig = px.line(x=np.arange(ds.trial_indices.shape[0]), y=ds.trial_indices, labels={'x': 'Time', 'y': 'Trial Indices'}, title='Trial Indices Over Time')
fig.show()

In [ ]:
# plot blocks with hover showing ds.block_labels
df_blocks = pd.DataFrame({
    'time': np.arange(ds.block_indices.shape[0]),
    'block_index': ds.block_indices,
    'block_label': [str(l) for l in ds.block_labels]
})

fig = px.line(
    df_blocks,
    x='time',
    y='block_index',
    hover_data=['block_label'],
    labels={'time': 'Time', 'block_index': 'Block Indices', 'block_label': 'Block Label'},
    title='Block Indices Over Time'
)
fig.update_traces(mode='lines+markers', marker=dict(size=6))
fig.show()

In [ ]:
# plot ds.behavioral_time
plt = px.line(x=np.arange(ds.behavioral_time.shape[0]), y=ds.behavioral_time, labels={'x': 'Time (samples)', 'y': 'Behavioral Time (ms)'}, title='Behavioral Time Over Samples')
plt.show()

In [ ]:
_ = plotting_neuronal_behavioural_plotly(ds.x.T.toarray(), ds.b.toarray().flatten(), b_names=list(ds.b_labels_dict.values()), b_colors=ds.get_color_map_for_plotting())